# 03b · Train ResNet-18: other ways to fight background bias
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/03b_train_methods.ipynb)

**Runtime → T4 GPU → Run all.** 3 конфигурации × 3 seed, ~1.5–2 ч. Всё как у `resnet_B`, меняется только то,
что делаем с фоном во время обучения:

- `resnet_gray` — с вероятностью 0.5 фон заменяется серым (просто убрать фон);
- `resnet_blur` — с вероятностью 0.5 собственный фон размывается (животное остаётся чётким);
- `resnet_cutmix` — фон не трогаем, обычная аугментация CutMix (p = 0.5 на батч): помогает ли
  общая аугментация так же, как целевая?

Учти при анализе: `resnet_gray` видит при обучении ровно тот же серый фон, что и тестовое условие fg-only.

Уже обученные запуски пропускаются, поэтому повторный Run all безопасен.
Если Colab отключился — просто запусти ноутбук снова, обучение продолжится с последней эпохи.
Checkpoints: `MyDrive/bgbias/runs/<name>_seed<k>/`. В конце истории обучения пушатся в GitHub.
После всех 03* ноутбуков — `04_evaluate`, затем `05_analysis`.

In [ ]:
CONFIGS = ['resnet_gray', 'resnet_blur', 'resnet_cutmix']
SEEDS = [0, 1, 2]

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
REPO_URL = "https://github.com/mundmina/dl_project.git"
REPO_DIR = "/content/dl_project"
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:  # always run the latest code from GitHub
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    from google.colab import userdata
    userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import pandas as pd
from bgbias.train import load_config, train

configs = {name: load_config(f"{REPO_DIR}/configs/{name}.yaml") for name in CONFIGS}
pd.DataFrame(configs)   # paired configs (X_A / X_B) must differ only in name and p_swap

In [ ]:
# seed-major order: if Colab stops early, every config already has seed 0
histories = {}
for seed in SEEDS:
    for name, cfg in configs.items():
        run_dir = RUNS / f"{name}_seed{seed}"
        print(f"=== {run_dir.name} ===")
        histories[(name, seed)] = pd.DataFrame(train(cfg, seed, data, run_dir))

In [ ]:
rows = []
for (name, seed), h in histories.items():
    best = h.loc[h.val_acc_original.idxmax()]
    rows.append({"model": name, "seed": seed, "best_epoch": int(best.epoch),
                 "val_original": best.val_acc_original, "val_swapped": best.val_acc_swapped,
                 "gap": best.val_acc_original - best.val_acc_swapped})
best = pd.DataFrame(rows)
print(best.groupby("model")[["val_original", "val_swapped", "gap"]].agg(["mean", "std"]).round(4))
best.round(4)

In [ ]:
from bgbias.github_sync import sync
sync(REPO_DIR, "Training histories: " + ", ".join(CONFIGS))